# ADL raw 데이터 → 하루 단위 사건·사고 보고서 프롬프트

`adl_raw_records`(데이터바우처 응급/사망 샘플)의 ADL 데이터를 **토큰 효율적인 압축
프롬프트**로 변환해 `.md` 파일로 출력한다. 프롬프트 1개는 대상자의 **하루**를 분석
대상으로 하고, 추세 판단을 돕도록 **직전 3일을 맥락**으로 함께 담는다. 각 파일은
`[SYSTEM]` 분석 지시문 + `[USER]` 압축 데이터로 구성되며, 그대로 LLM 에 붙여넣으면
그날 사건·사고가 일어났는지 보고하거나 가까운 시일 내 발생 가능성을 예측하는
**사건·사고 보고서**를 받을 수 있다. 이 노트북은 LLM 을 직접 호출하지 않는다.

`adl_raw_records` 의 5개 배열 컬럼은 분 단위 1440개 시계열이라 raw 로 넘기면 한 행만
~7,200개 정수다. 위치 시계열은 런렝스 세그먼트 타임라인으로, 시간별 배열은 시간대
버킷으로 압축한다.

**실행 순서**: ① `adl_raw_ingest.ipynb` 로 `adl_raw_records` 적재 → ② 이 노트북을
위에서 아래로 실행 → `notebooks/llm_prompts/` 에 (대상자×날짜)별 프롬프트 파일 생성.

In [1]:
import os
import re
import sys
from pathlib import Path

# 작업 디렉토리를 프로젝트 루트로 이동 (.env.local·app import 해석용, 폴더 생성 아님).
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from app.config import settings

print(f"프로젝트 루트: {ROOT}")

프로젝트 루트: C:\Users\kdwkd\documents\project\salpyeobom\salpyeobom-backend


In [2]:
# app/database.py 의 TORTOISE_ORM 재사용. 읽기 전용이라 init 만 호출.
from tortoise import Tortoise

from app.database import TORTOISE_ORM

if not Tortoise._inited:
    await Tortoise.init(config=TORTOISE_ORM)

_url = settings.DATABASE_URL
print("DB 연결 완료:", _url.split("@")[-1] if "@" in _url else _url)

DB 연결 완료: 127.0.0.1:5432/salpyeobom


## 압축 변환 전략

분 단위 1440 배열을 **런렝스 세그먼트**로 압축한다. 통계 피처만 뽑으면 순서·전이
정보(예: '새벽 3시 욕실 이동 후 4시간 무활동')가 사라지는데, 이 시퀀스가 사건·사고
신호의 핵심이다. 런렝스는 체류 구간을 한 줄씩 보존하면서 토큰을 크게 줄인다. 10분
미만 구간은 직전 구간에 흡수하고 인접한 같은 위치는 하나로 합쳐 세그먼트 수를
줄인다. 24개 시간별 배열은 4개 시간대 버킷으로 집계한다. 깨진 필드
(`night_aix_ratio`, HH:MM 아닌 수면 시각 등)는 변환 단계에서 걸러내거나 '데이터
주의'로 표기한다.

In [3]:
SENTINEL_VALUES = {254, 255}  # '해당 없음'·미측정 표식 (실측값 아님)
HOUR_BUCKETS = [("심야", 0, 6), ("오전", 6, 12), ("오후", 12, 18), ("저녁", 18, 24)]


def cv(rec: dict, key: str):
    """rec[key] 를 읽되 pandas NaN/NaT 는 None 으로 정규화."""
    v = rec.get(key)
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass  # 리스트 등 — NaN 아님
    return v


def mins_to_hhmm(minute: int) -> str:
    """0~1440 분 인덱스를 'HH:MM' 로 변환."""
    minute = max(0, min(1440, int(minute)))
    return f"{minute // 60:02d}:{minute % 60:02d}"


def runlength_segments(values: list, *, min_minutes: int = 10) -> list[tuple[int, int, int]]:
    """분 단위 배열을 (시작분, 끝분, 대표값) 세그먼트로 압축.

    짧은 구간(min_minutes 미만)은 직전에 흡수하고, 흡수로 인접하게 된 같은 값
    세그먼트는 하나로 병합한다.
    """
    if not values:
        return []
    raw: list[list[int]] = []
    start = 0
    for i in range(1, len(values) + 1):
        if i == len(values) or values[i] != values[start]:
            raw.append([start, i, values[start]])
            start = i
    merged: list[list[int]] = []
    for seg in raw:
        if merged and (seg[2] == merged[-1][2] or seg[1] - seg[0] < min_minutes):
            merged[-1][1] = seg[1]  # 동일값이거나 짧은 구간 → 직전에 흡수·병합
        else:
            merged.append(seg)
    return [(s, e, v) for s, e, v in merged]


def parse_clock(v) -> str | None:
    """'22:30' 같은 정상 HH:MM 만 반환. '45'·'38' 등 깨진 값은 None."""
    s = str(v).strip() if v is not None else ""
    if ":" in s:
        a, _, b = s.partition(":")
        if a.isdigit() and b.isdigit() and 0 <= int(a) < 24 and 0 <= int(b) < 60:
            return f"{int(a):02d}:{int(b):02d}"
    return None


def bucket_hourly(values: list) -> dict:
    """24개 시간별 배열을 4개 시간대 버킷의 합으로 집계."""
    if not values or len(values) < 24:
        return {}
    out = {}
    for name, lo, hi in HOUR_BUCKETS:
        chunk = [v for v in values[lo:hi] if v is not None]
        if chunk:
            out[name] = sum(chunk)
    return out


def _zero_run_hours(window: list) -> float:
    """활동지수 0 이 이어진 최장 연속 시간(h)."""
    longest = cur = 0
    for a in window:
        cur = cur + 1 if a == 0 else 0
        longest = max(longest, cur)
    return longest / 60


def summarize_place_timeline(place, aix_1, *, max_segments: int = 20) -> str | None:
    """place_code 분 배열을 위치 세그먼트 타임라인으로 압축. 구간별 활동지수 종속 집계."""
    if not place:
        return None
    segs = runlength_segments(place, min_minutes=10)
    lines = []
    for start, end, code_ in segs[:max_segments]:
        label = f"{mins_to_hhmm(start)}-{mins_to_hhmm(end)} 위치{code_}"
        window = [a for a in (aix_1[start:end] if aix_1 else []) if a is not None]
        if window:
            label += f" · 활동 평균{sum(window) / len(window):.0f}(최대{max(window)})"
            run = _zero_run_hours(window)
            if run >= 1:
                label += f" · 무활동연속 {run:.1f}h"
        lines.append("- " + label)
    if len(segs) > max_segments:
        lines.append(f"- … (외 {len(segs) - max_segments}개 구간 생략)")
    return "\n".join(lines)


def summarize_outgoing(outgoing) -> str | None:
    """outgoing 분 배열에서 sentinel 아닌 구간을 외출 이벤트로 추출."""
    if not outgoing:
        return None
    intervals, start = [], None
    for i, v in enumerate(outgoing):
        if v not in SENTINEL_VALUES and start is None:
            start = i
        elif v in SENTINEL_VALUES and start is not None:
            intervals.append((start, i))
            start = None
    if start is not None:
        intervals.append((start, len(outgoing)))
    intervals = [(s, e) for s, e in intervals if e - s >= 10]  # 10분 미만 잡음 제거
    if not intervals:
        return "외출 기록 없음"
    return "; ".join(f"{mins_to_hhmm(s)}-{mins_to_hhmm(e)}({e - s}분)" for s, e in intervals)


def summarize_sleep_segments(sleep_depth) -> str | None:
    """sleep_depth 분 배열에서 주 수면 구간을 추출."""
    if not sleep_depth:
        return None
    asleep = [(s, e, v) for s, e, v in runlength_segments(sleep_depth, min_minutes=15) if v]
    if not asleep:
        return None
    s, e, _ = max(asleep, key=lambda x: x[1] - x[0])
    dur = e - s
    return f"주 수면 구간 {mins_to_hhmm(s)}-{mins_to_hhmm(e)} ({dur // 60}시간 {dur % 60}분), 수면 블록 {len(asleep)}개"


def _range_line(label, values, unit) -> str | None:
    pairs = [(h, v) for h, v in enumerate(values or []) if v is not None and v == v]
    if not pairs:
        return None
    vs = [v for _, v in pairs]
    peak_h = max(pairs, key=lambda p: p[1])[0]
    return f"{label} {min(vs):.1f}~{max(vs):.1f}{unit} (평균 {sum(vs) / len(vs):.1f}, 최고 {peak_h}시)"


def summarize_environment(temp, humi, illu) -> str | None:
    """시간별 온/습도/조도 배열을 'min~max(평균, 피크시각)' 한 줄로."""
    parts = [_range_line("온도", temp, "℃"), _range_line("습도", humi, "%"),
             _range_line("조도", illu, "lux")]
    parts = [p for p in parts if p]
    return " · ".join(parts) if parts else None


print("압축 변환 함수 정의 완료")

압축 변환 함수 정의 완료


In [4]:
from dataclasses import dataclass, field


@dataclass
class DayCard:
    """대상자 하루치 ADL 을 LLM 친화적 마크다운 카드로 담는 정규화 구조."""
    date: str
    source: str  # '응급' | '사망'
    is_event_day: bool = False
    is_target: bool = False  # 이번 보고서의 분석 대상일 여부
    event_note: str | None = None
    place_timeline: str | None = None
    outgoing_line: str | None = None
    sleep_line: str | None = None
    bath_line: str | None = None
    activity_line: str | None = None
    environment_line: str | None = None
    notes: list = field(default_factory=list)  # 데이터 품질 경고


def format_day_card(card: DayCard) -> str:
    """DayCard → Markdown 일별 카드 텍스트."""
    head = f"## {card.date}"
    if card.is_event_day:
        head += f"  [이벤트 발생일: {card.source}]"
    if card.is_target:
        head += "  ← 분석 대상일"
    parts = [head]
    if card.event_note:
        parts.append(f"이벤트 기록: {card.event_note}")
    if card.place_timeline:
        parts.append("위치 타임라인:\n" + card.place_timeline)
    for label, value in [("외출", card.outgoing_line), ("수면", card.sleep_line),
                         ("목욕", card.bath_line), ("활동지수", card.activity_line),
                         ("환경", card.environment_line)]:
        if value:
            parts.append(f"{label}: {value}")
    if card.notes:
        parts.append("데이터 주의: " + "; ".join(card.notes))
    return "\n".join(parts)


def raw_record_to_daycard(rec: dict) -> DayCard:
    """adl_raw_records 한 행(dict) → DayCard."""
    notes: list = []
    date = cv(rec, "lifeog_date")
    is_event = cv(rec, "emergency_date") is not None or cv(rec, "death_date") is not None
    note = cv(rec, "emergency_record") or cv(rec, "death_record")

    # 수면 — sleep_depth 타임라인 + 스칼라, 깨진 시각값 방어
    sleep_bits = []
    if seg := summarize_sleep_segments(cv(rec, "sleep_depth_1_list")):
        sleep_bits.append(seg)
    if (period := cv(rec, "total_sleep_period")) is not None:
        sleep_bits.append(f"총 수면 기록값 {period:.0f}분")
    if cv(rec, "sleep_start_time_d") and not parse_clock(cv(rec, "sleep_start_time_d")):
        notes.append("수면 시작/종료 시각 원본값이 HH:MM 형식이 아니라 추론에서 제외")

    # 목욕
    bath_bits = []
    if (n := cv(rec, "bath_count_d")) is not None:
        bath_bits.append(f"{int(n)}회")
    if (t := cv(rec, "bath_time_d")) is not None:
        bath_bits.append(f"총 {t:.0f}분")
    if cv(rec, "bath_count_in_sleep"):
        bath_bits.append(f"수면 중 {int(cv(rec, 'bath_count_in_sleep'))}회")
    if (nm := cv(rec, "bath_nomove_time")) is not None:
        bath_bits.append(f"무동작 {nm:.0f}분")

    # 외출 — 분단위 타임라인 + 일집계 스칼라
    out_seg = summarize_outgoing(cv(rec, "outgoing_1_list"))
    out_bits = []
    if (n := cv(rec, "outgoing_count_d")) is not None:
        out_bits.append(f"{int(n)}회")
    if (t := cv(rec, "outgoing_time_d")) is not None:
        out_bits.append(f"총 {t:.0f}분")
    if cv(rec, "outgoing_late_night_count_d"):
        out_bits.append(f"심야 {int(cv(rec, 'outgoing_late_night_count_d'))}회")
    out_scalar = ", ".join(out_bits)
    if out_seg and out_scalar:
        outgoing_line = f"{out_seg} / 일집계 {out_scalar}"
    else:
        outgoing_line = out_seg or (f"일집계 {out_scalar}" if out_scalar else None)

    # 활동지수 — aix_h 시간대 버킷 + 스칼라. night_aix_ratio 는 값 범위 비정상 → 제외
    act_bits = []
    if buckets := bucket_hourly(cv(rec, "aix_h_list")):
        act_bits.append(" · ".join(f"{k} 합{v}" for k, v in buckets.items()))
    if (d := cv(rec, "aix_d")) is not None:
        act_bits.append(f"일활동지수 {d:.0f}")
    if cv(rec, "aix_1_eq_0_repeat_count"):
        act_bits.append(f"활동0 반복 {int(cv(rec, 'aix_1_eq_0_repeat_count'))}회")
    if cv(rec, "night_aix_ratio") is not None:
        notes.append("night_aix_ratio 등 일부 비율 지표는 값 범위가 비정상이라 카드에서 제외")

    return DayCard(
        date=date.isoformat() if date else "날짜 미상",
        source=cv(rec, "source_type") or "?",
        is_event_day=is_event,
        event_note=note.replace("\n", " ").strip()[:300] if note else None,
        place_timeline=summarize_place_timeline(
            cv(rec, "place_code_1_list"), cv(rec, "aix_1_list")
        ),
        outgoing_line=outgoing_line,
        sleep_line=" / ".join(sleep_bits) or None,
        bath_line=", ".join(bath_bits) or None,
        activity_line=" · ".join(act_bits) or None,
        environment_line=summarize_environment(
            cv(rec, "temp_list"), cv(rec, "humi_list"), cv(rec, "illu_list")
        ),
        notes=notes,
    )


print("DayCard 정의 완료")

DayCard 정의 완료


In [5]:
from app.models.adl_raw import AdlRawRecord

raw_df = pd.DataFrame(await AdlRawRecord.all().values())
if raw_df.empty:
    raise RuntimeError("adl_raw_records 가 비어 있습니다. 먼저 adl_raw_ingest.ipynb 로 적재하세요.")

# 인구통계 컬럼은 엑셀 병합셀이라 그룹 첫 행에만 존재 → care_recipient_id 그룹 내 전파
DEMO_COLS = ["age", "sex", "alone", "house_structure", "bath_location"]
for col in DEMO_COLS:
    g = raw_df.groupby("care_recipient_id")[col]
    raw_df[col] = g.ffill().groupby(raw_df["care_recipient_id"]).bfill()

patient_ids = sorted(raw_df["care_recipient_id"].dropna().unique())
print(f"adl_raw_records: {len(raw_df)}행, 대상자 {len(patient_ids)}명")

CONTEXT_DAYS = 3  # 분석 대상일 직전 며칠을 맥락으로 첨부


def patient_rows(cid) -> list[dict]:
    """한 대상자의 모든 날을 lifeog_date 오름차순 dict 리스트로."""
    return (raw_df[raw_df["care_recipient_id"] == cid]
            .sort_values("lifeog_date", na_position="last").to_dict("records"))


def _patient_header(first: dict) -> str:
    """대상자 인구통계 헤더 한 줄."""
    demo = [str(first.get("care_recipient_id"))]
    if cv(first, "age") is not None:
        demo.append(f"{int(cv(first, 'age'))}세")
    if sex := {"M": "남성", "F": "여성"}.get(cv(first, "sex")):
        demo.append(sex)
    if cv(first, "alone") == "Y":
        demo.append("독거")
    if cv(first, "house_structure"):
        demo.append(str(cv(first, "house_structure")))
    if cv(first, "bath_location"):
        demo.append(f"욕실 {cv(first, 'bath_location')}")
    return "# 대상자 " + " · ".join(demo)


def build_day_prompt(rows: list[dict], i: int) -> str:
    """rows[i] 를 분석 대상으로, 직전 CONTEXT_DAYS 일을 맥락으로 묶은 압축 프롬프트."""
    window = rows[max(0, i - CONTEXT_DAYS):i + 1]
    cards = [raw_record_to_daycard(r) for r in window]
    cards[-1].is_target = True  # window 마지막 = 분석 대상일
    body = "\n\n---\n\n".join(format_day_card(c) for c in cards)
    return _patient_header(rows[0]) + "\n\n" + body


# 검증 — 이벤트 발생일을 분석 대상으로 한 프롬프트를 출력해 육안 확인
_rows = patient_rows(patient_ids[0])
_evt = next((j for j, r in enumerate(_rows)
             if cv(r, "emergency_date") or cv(r, "death_date")), len(_rows) - 1)
_sample = build_day_prompt(_rows, _evt)
print(_sample)
print("\n" + "─" * 60)
print(f"문자 수: {len(_sample):,}  /  추정 토큰(대략 len/2): ~{len(_sample) // 2:,}")

adl_raw_records: 60행, 대상자 2명
# 대상자 302405.0 · 85세 · 남성 · 독거 · 주택 · 욕실 옥내

## 2023-03-26
위치 타임라인:
- 00:00-04:10 위치30 · 활동 평균29(최대400)
- 04:10-04:58 위치10 · 활동 평균270(최대1000)
- 04:58-05:34 위치30 · 활동 평균6(최대66)
- 05:34-07:13 위치254 · 활동 평균36(최대666)
- 07:13-09:49 위치30 · 활동 평균182(최대1000)
- 09:49-10:48 위치10 · 활동 평균423(최대1000)
- 10:48-11:34 위치30 · 활동 평균174(최대1000)
- 11:34-12:04 위치254 · 활동 평균0(최대0)
- 12:04-12:19 위치30 · 활동 평균186(최대833)
- 12:19-12:29 위치254 · 활동 평균20(최대133)
- 12:29-12:39 위치30 · 활동 평균167(최대800)
- 12:39-13:00 위치254 · 활동 평균135(최대633)
- 13:00-14:48 위치10 · 활동 평균331(최대1000)
- 14:48-15:44 위치30 · 활동 평균100(최대1000)
- 15:44-15:54 위치254 · 활동 평균23(최대233)
- 15:54-18:14 위치30 · 활동 평균97(최대800)
- 18:14-18:39 위치254 · 활동 평균9(최대133)
- 18:39-21:05 위치30 · 활동 평균397(최대1000)
- 21:05-21:50 위치10 · 활동 평균529(최대1000)
- 21:50-22:58 위치30 · 활동 평균288(최대866)
- … (외 2개 구간 생략)
외출: 외출 기록 없음 / 일집계 8회, 총 167분
수면: 총 수면 기록값 0분
목욕: 0회, 총 0분, 무동작 0분
활동지수: 심야 합336 · 오전 합1181 · 오후 합1094 · 저녁 합1805 · 일활동지수 195
환경: 온도 23.3~26.9℃ (평

In [6]:
# LLM 에게 줄 분석 지시문. 출력 파일의 [SYSTEM] 섹션에 그대로 들어간다.
SYSTEM_PROMPT = """당신은 고령자 원격 모니터링 데이터를 분석해 사건·사고를 예측·보고하는 안전 분석 전문가입니다.

입력은 한 대상자의 며칠치 '일별 카드'입니다. 시간 오름차순이며, 마지막 카드(헤더에
'← 분석 대상일' 표기)가 이번 보고서의 분석 대상입니다. 앞의 카드들은 추세 판단을
돕는 직전 며칠 맥락입니다. 분석 대상일이 [이벤트 발생일]로 표시되면 그날 응급 또는
사망 사건이 이미 발생한 것입니다.

[보고 모드 — 분석 대상일 기준 자동 선택]
- 사후 보고: 분석 대상일이 [이벤트 발생일]이면 그 사건을 보고하고, 맥락 카드와 함께
  사건에 앞선 전조(경고 신호)를 짚습니다.
- 사전 예측: 분석 대상일이 [이벤트 발생일]이 아니면, 그날 ADL 패턴과 직전 며칠
  추세로 가까운 시일 내 사건·사고 발생 가능성을 예측합니다.

[카드 용어]
- 위치 타임라인: 'HH:MM-HH:MM 위치{코드}' = 해당 구간 동안 그 위치에 머무름.
  코드 0 은 주생활공간(거실·안방 등)으로 추정되나 단정하지 마십시오.
  10분 미만 짧은 구간은 센서 잡음 제거를 위해 인접 구간에 병합돼 있습니다.
- 활동지수: 분 단위 움직임 강도(0 = 무활동). '무활동연속'은 활동지수 0이 이어진 시간.
- 외출: 집 밖에 있던 시간 구간. 값 254·255 는 '해당 없음' 표식이며 실측값이 아닙니다.
- 시간대 활동(심야 0-5시 / 오전 6-11시 / 오후 12-17시 / 저녁 18-23시): 시간별 활동지수 합.
- 환경: 실내 온도·습도·조도의 하루 범위.

[데이터 품질 경고]
- night_aix_ratio 등 일부 비율 지표는 값 범위가 비정상이어서 카드에서 제외했습니다.
- '데이터 주의'로 표시된 항목은 신뢰도가 낮으니 추론 근거로 쓰지 마십시오.

[판단할 사건·사고 유형]
주요 유형(예시): 낙상, 응급 상황(장시간 무동작·의식 저하 의심), 장기 외출·실종,
고독사 위험, 야간 이상행동(심야 배회·욕실 장시간 체류), 위생·건강 악화.
위 목록에 없는 이상 징후가 보이면 자유롭게 새 유형으로 지목하십시오.

[위험도 등급]
1단계 정상 — 우려할 패턴 없음
2단계 주의 — 경미한 변화, 경과 관찰 필요
3단계 경고 — 뚜렷한 악화 추세, 단기 개입 권고
4단계 위험 — 사건 발생 또는 임박, 즉시 대응 필요

[출력 형식]
한국어 마크다운. 분석 대상일 하루에 대한 보고서입니다.

# 사건·사고 보고서 — {대상자 정보} · {분석 대상일}

판정: 사건 발생 또는 발생 위험 예측 중 하나
위험도: {등급명} ({n}/4 단계)
사건 유형: {유형 — 예측 시 가장 가능성 높은 것}
발생·예상 시점: {날짜·시간대 — 예측 시 '수일 내' 등 시급성}

## 경위 및 근거
분석 대상일의 수면·활동·외출·욕실 패턴을 직전 며칠 맥락과 비교해 판정을
뒷받침하십시오. 구체적 수치·시각과 함께 제시하십시오.

## 권고 조치
- 즉시: ...
- 단기: ...

[원칙]
의학적 진단이나 단정적 표현은 피하고 관찰된 행동 변화에 근거해 기술하십시오.
데이터에 없는 사실을 지어내지 마십시오. 근거가 약하면 불확실성을 명시하되,
고령자 안전 특성상 위험 신호를 과소평가하지 마십시오."""

print(f"SYSTEM_PROMPT 길이: {len(SYSTEM_PROMPT):,}자")

SYSTEM_PROMPT 길이: 1,580자


In [7]:
PROMPT_DIR = ROOT / "notebooks" / "llm_prompts"
PROMPT_DIR.mkdir(exist_ok=True)
for _old in PROMPT_DIR.glob("*.md"):  # 이전 실행 산출물 정리
    _old.unlink()


def build_full_prompt(rows: list[dict], i: int) -> str:
    """[SYSTEM] 분석 지시문 + [USER] 압축 데이터 = 그대로 LLM 에 붙여넣을 완성 프롬프트."""
    return f"[SYSTEM]\n{SYSTEM_PROMPT}\n\n[USER]\n{build_day_prompt(rows, i)}\n"


def write_day_prompt(rows: list[dict], i: int) -> Path:
    """rows[i] 하루를 분석 대상으로 한 완성 프롬프트를 .md 파일로 저장."""
    rec = rows[i]
    src = rec.get("source_type") or "기타"
    cid = re.sub(r"[^0-9A-Za-z가-힣_-]", "", str(rec.get("care_recipient_id")).replace(".", "_"))
    date = cv(rec, "lifeog_date")
    date_s = date.isoformat() if date else f"day{i:02d}"
    path = PROMPT_DIR / f"{src}_{cid}_{date_s}.md"
    path.write_text(build_full_prompt(rows, i), encoding="utf-8")
    return path


written = []
for cid in patient_ids:
    rows = patient_rows(cid)
    written += [write_day_prompt(rows, i) for i in range(len(rows))]

total_kb = sum(p.stat().st_size for p in written) / 1024
print(f"{len(written)}개 하루 단위 프롬프트 파일 생성 → {PROMPT_DIR}")
print(f"총 {total_kb:.1f} KB / 파일당 평균 추정 토큰 ~{int(total_kb * 1024 / len(written) / 2):,}")

60개 하루 단위 프롬프트 파일 생성 → C:\Users\kdwkd\documents\project\salpyeobom\salpyeobom-backend\notebooks\llm_prompts
총 590.3 KB / 파일당 평균 추정 토큰 ~5,037


In [8]:
await Tortoise.close_connections()
print("DB 연결 종료")

DB 연결 종료
